# Análise qualitativa de radiografias OOF — TP1 E2

## Objetivo e seleção dos casos

Montagem de oito radiografias de HOG + Gradient Boosting, com sexo como covariável: os quatro menores e os quatro maiores erros absolutos entre 2.800 previsões OOF no treino. IDs, referências, previsões e dobras foram conferidos contra o pacote de resultados e estão incorporados à célula.

Cada caso foi predito pelo modelo da dobra que não o utilizou no treinamento. A seleção pelos extremos é ilustrativa e não constitui amostra aleatória. O protocolo é por imagem, sem independência por paciente verificada. Nenhum caso do teste externo é utilizado.

## Execução no Google Colab

1. Conectar um ambiente com CPU e executar a célula de código.
2. Não é necessário carregar os ZIPs de resultados. As imagens são reutilizadas do cache quando disponíveis; caso contrário, a rotina obtém a versão 2 da base pelo KaggleHub (aproximadamente 9,3 GB).
3. Ao concluir, salvar `radiografias_analise_oof_TP1.zip`. O pacote inclui a figura em PNG/PDF, os oito PNGs processados, a tabela de casos, a legenda e a proveniência. Caso o download automático não abra, o arquivo pode ser localizado pelo painel **Arquivos**, no caminho impresso na saída.

A célula também pode ser executada ao final de outro notebook, na sessão atual. Não depende de variáveis anteriores, não ajusta modelos e não repete a validação cruzada. As porcentagens representam marcos desta etapa. As figuras documentam a análise de desenvolvimento e não substituem a avaliação final.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import io
import json
import shutil
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from google.colab import files

print("0% — Preparando oito radiografias; não haverá treinamento.", flush=True)
SOURCE_ZIP_SHA = '6c95a1d3c6f1ccaf9cfe07f9b62a8d763132e81a308c2390ebaee0855461c4c3'
SOURCE_OOF_SHA = '5e1c67eb8f8a25e22a04b6252ce1c9d095287b0bab4c58a4a3e318333dfacd7c'
CASES_CSV = 'id,family,model,fold,boneage,y_pred,abs_error,male,example\n4220,HOG,GradientBoosting,1,150.0,149.9351961853089,0.0648038146910892,1,Menor erro\n14986,HOG,GradientBoosting,1,120.0,119.93383040940954,0.0661695905904622,0,Menor erro\n4869,HOG,GradientBoosting,3,138.0,138.09870001031564,0.098700010315639,0,Menor erro\n1574,HOG,GradientBoosting,2,132.0,132.11923124746488,0.1192312474648815,1,Menor erro\n3337,HOG,GradientBoosting,2,18.0,121.39729978274708,103.39729978274708,1,Maior erro\n8460,HOG,GradientBoosting,1,36.0,136.0526793863786,100.0526793863786,1,Maior erro\n4199,HOG,GradientBoosting,2,228.0,128.17302004472205,99.82697995527796,1,Maior erro\n5047,HOG,GradientBoosting,3,24.0,121.47237526038856,97.47237526038856,1,Maior erro\n'
PREPROCESS_CODE = '"""Imagem de entrada comum aos descritores do TP1.\n\nUse preprocess_image no HOG, na textura e na intensidade para que as três\nfamílias vejam os mesmos pixels. Não há recorte, remoção de fundo, realce\nde contraste nem parâmetros ajustados com imagens de validação ou teste.\n"""\n\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\n\nIMAGE_SIZE = (224, 224)\n\n\ndef preprocess_image(image_path: str | Path, *, size: tuple[int, int] = IMAGE_SIZE) -> np.ndarray:\n    """Lê PNG, converte a cinza, redimensiona e devolve float32 em [0, 1]."""\n    with Image.open(image_path) as source:\n        gray = source.convert("L").resize(size, Image.Resampling.LANCZOS)\n        pixels = np.asarray(gray, dtype=np.uint8)\n    return pixels.astype(np.float32) / 255.0\n'
cases = pd.read_csv(io.StringIO(CASES_CSV), dtype={"id": str})
if len(cases) != 8 or cases.id.nunique() != 8:
    raise ValueError("A lista deve conter oito IDs distintos.")
if not np.isfinite(cases[["boneage", "y_pred", "abs_error"]]).all().all():
    raise ValueError("Valores não finitos nos exemplos.")
if not np.allclose(np.abs(cases.y_pred - cases.boneage), cases.abs_error, rtol=0, atol=1e-9):
    raise ValueError("Erros diferentes das previsões conferidas.")

# Reutiliza somente pastas conhecidas deste dataset. Não depende de variáveis antigas.
roots = [Path("/root/.cache/kagglehub/datasets/kmader/rsna-bone-age/versions/2"),
         Path("/kaggle/input/rsna-bone-age")]
image_dir = None
for root in roots:
    if root.is_dir():
        for first in sorted(root.rglob(str(cases.id.iloc[0]) + ".png")):
            if all((first.parent / (image_id + ".png")).is_file() for image_id in cases.id):
                image_dir = first.parent
                break
    if image_dir is not None:
        break
if image_dir is None:
    print("As imagens não estão no cache desta sessão. Obtendo o dataset versão 2.", flush=True)
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub==1.0.2"], check=True)
        import kagglehub
    dataset = Path(kagglehub.dataset_download("kmader/rsna-bone-age/versions/2"))
    matches = [p.parent for p in dataset.rglob(str(cases.id.iloc[0]) + ".png")
               if all((p.parent / (image_id + ".png")).is_file() for image_id in cases.id)]
    if len(matches) != 1:
        raise FileNotFoundError("Não foi localizada uma pasta única com as oito radiografias.")
    image_dir = matches[0]
print("25% — Radiografias disponíveis em:", image_dir, flush=True)

work = Path(tempfile.mkdtemp(prefix="radiografias_oof_tp1_", dir="/content"))
out = work / "resultados"
out.mkdir()
processed = out / "imagens_comuns_224"
processed.mkdir()
# Usa exatamente o código de pré-processamento presente no ZIP já conferido.
namespace = {}
exec(compile(PREPROCESS_CODE, "image_preprocessing.py", "exec"), namespace)
preprocess_image = namespace["preprocess_image"]
(out / "image_preprocessing.py").write_text(PREPROCESS_CODE, encoding="utf-8")

pixels_by_id = {}
sources = []
for number, row in enumerate(cases.itertuples(), 1):
    path = image_dir / (row.id + ".png")
    with Image.open(path) as original:
        original_size = list(original.size)
    pixels = preprocess_image(path)
    if pixels.shape != (224, 224) or not np.isfinite(pixels).all():
        raise ValueError("Imagem processada inválida: " + row.id)
    pixels_by_id[row.id] = pixels
    # Guarda o PNG de 8 bits que corresponde aos pixels apresentados ao descritor.
    Image.fromarray(np.rint(pixels * 255).astype(np.uint8)).save(processed / (row.id + ".png"))
    sources.append({"id": row.id, "source_png_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                    "source_size": original_size})
    print(f"Radiografias: {number}/8", flush=True)
print("50% — Oito imagens processadas com o protocolo comum.", flush=True)

plt.rcParams.update({"font.size": 10, "pdf.fonttype": 42})
fig, axes = plt.subplots(2, 4, figsize=(10.8, 7.4), layout="constrained")
for axis, row in zip(axes.flat, cases.itertuples()):
    axis.imshow(pixels_by_id[row.id], cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"{row.example} · ID {row.id} · dobra {row.fold}\n"
                   f"Referência {row.boneage:.0f} · predito {row.y_pred:.2f}\n"
                   f"Erro absoluto {row.abs_error:.2f} meses", fontsize=9)
    axis.axis("off")
fig.suptitle("HOG + Gradient Boosting com sexo: exemplos OOF no treino\n"
             "Acima: quatro menores erros; abaixo: quatro maiores erros", fontsize=11)
fig.savefig(out / "figura_radiografias_oof_HOG_GB.png", dpi=200)
fig.savefig(out / "figura_radiografias_oof_HOG_GB.pdf")
plt.show()
plt.close(fig)
cases.to_csv(out / "casos_radiografias_oof.csv", index=False)

manifest = {"source_results_zip_sha256": SOURCE_ZIP_SHA,
            "source_predictions_oof_sha256": SOURCE_OOF_SHA,
            "dataset": "kmader/rsna-bone-age/versions/2",
            "family": "HOG", "model": "GradientBoosting", "sex_covariate": True,
            "partition": "OOF no treino de 2800 imagens", "patient_independence_verified": False,
            "selection": "4 menores e 4 maiores erros absolutos; desempate por ID como texto",
            "selection_pool_size": 2800, "new_model_fits": 0, "external_test_used": False,
            "preprocessing": "L; 224x224 Lanczos; float32/255; sem ROI/fundo/realce",
            "preprocessing_code_sha256": hashlib.sha256(PREPROCESS_CODE.encode()).hexdigest(),
            "images": sources}
(out / "proveniencia_radiografias.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
caption = ("Radiografias com os quatro menores erros (linha superior) e quatro maiores erros "
           "(linha inferior) entre 2.800 previsões OOF de HOG + Gradient Boosting, com sexo "
           "como covariável. Cada caso foi predito pelo modelo da dobra que não o utilizou "
           "no treinamento. Imagens exibidas após o pré-processamento comum de 224×224 "
           "pixels. Idades e erros em meses. A seleção é ilustrativa, usa extremos do erro "
           "e não representa uma amostra aleatória nem avaliação do teste final. "
           "Independência entre pacientes não verificada.")
(out / "legenda_para_artigo.txt").write_text(caption + "\n", encoding="utf-8")
(out / "LEIA_RADIOGRAFIAS.md").write_text(
    "# Radiografias OOF — TP1 E2\n\n" + caption +
    "\n\nOs IDs, referências e previsões foram selecionados do ZIP real já conferido. "
    "Este notebook apenas recupera as imagens e monta a figura; não refaz CV nem "
    "ajusta modelos. O SHA-256 do pacote de origem está em proveniencia_radiografias.json. "
    "Os casos 3337, 8460 e 5047 apresentam superestimação, e 4199, subestimação. "
    "Não atribuir uma causa anatômica a esses erros sem inspecionar as radiografias. "
    "Os exemplos de menor erro são 4220, 14986, 4869 e 1574.\n",
    encoding="utf-8")
print("75% — Figura, tabela, legenda e proveniência geradas.", flush=True)
archive = Path(shutil.make_archive(str(work / "radiografias_analise_oof_TP1"), "zip", out))
with ZipFile(archive) as check:
    output_files = [item for item in check.infolist() if not item.is_dir()]
    if check.testzip() is not None or len(output_files) != 15:
        raise ValueError("Falha na conferência do pacote de radiografias.")
print("100% desta etapa — radiografias_analise_oof_TP1.zip pronto.", flush=True)
print("Arquivo:", archive, flush=True)
print("SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest(), flush=True)
print("O artigo e a consolidação da avaliação final continuam pendentes.", flush=True)
files.download(str(archive))
